# K=5 manuscript figures — iteration notebook

Each figure has its own cell with the code inlined; tweak any line and re-run the cell. The shared `setup()` cell at the top runs the model and caches embeddings/states/diagnostics, so individual figure cells are cheap to re-run.

All figures save into `paper_5clusters/manuscript/figures/`.

**Run with**: `samenv` conda env + `KMP_DUPLICATE_LIB_OK=TRUE`.

The companion script is [figures_paper_5c.py](figures_paper_5c.py); helper functions live in [`paper/workflow_paper.py`](../../paper/workflow_paper.py).

In [ ]:
import os
import sys

# Force all paths to be absolute and explicit.
ROOT = "/Users/sophieabramian/Documents/troposwhere"
PAPER_K7 = os.path.join(ROOT, "paper")
PAPER_K5 = os.path.join(ROOT, "paper_5clusters")
FIGURES_OUT = os.path.join(PAPER_K5, "manuscript", "figures")

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
sys.path.insert(0, PAPER_K7)  # shared helpers (workflow_paper.py)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from scipy.ndimage import gaussian_filter1d

import workflow_paper as wfp


NUM_STATES = 5
MODEL_PATH_K5 = os.path.join(
    ROOT, "data", "runs", "exp_20260223_1140_5c129e", "best_model.pt"
)

# 5-colour palette (same as figures_5clusters.ipynb).
PALETTE = ["#1e78b3", "#e02222", "#bcbcdd", "#ffe066", "#5ec962"]
STATE_COLORS = np.array(PALETTE)
STATE_CMAP = ListedColormap(STATE_COLORS)
STATE_NORM = Normalize(vmin=0, vmax=NUM_STATES - 1)

wfp.NUM_STATES = NUM_STATES
wfp.MODEL_PATH = MODEL_PATH_K5
wfp.state_colors = STATE_COLORS
wfp.cluster_cmap = STATE_CMAP
wfp.norm_state = STATE_NORM
wfp.FIGURES_DIR = FIGURES_OUT
wfp.FIGURES_SI_DIR = FIGURES_OUT

assert FIGURES_OUT.startswith(PAPER_K5), "Figure output must stay inside paper_5clusters/"
os.makedirs(FIGURES_OUT, exist_ok=True)

# Shared style constants (mirror K=7 notebook).
INTERVALS = [(10000, 10300), (5000, 5300)]  # same time-index trajectories as K=7
TRAJ_COLORS = ["k", "k"]
TRAJ_STYLES = ["--", ":"]
TRAJ_SMOOTH_WIN = 7


# ---------------------------------------------------------------------------
# Shared setup — cached lazily
# ---------------------------------------------------------------------------

## Setup

Loads the K=5 model, runs inference, loads diagnostics. Cached in `_cache` so re-running is a no-op.

In [ ]:
_cache = {}

def setup():
    if "states" in _cache:
        return _cache

    print("[setup] loading dataloader + K=5 model...")
    loader, input_dim = wfp.load_dataloader()
    model = wfp.load_model(input_dim=input_dim)
    out = wfp.extract_latents(model, loader)
    embeddings = out["embeddings"]
    states = out["states"]
    emb_pca, pca = wfp.run_pca(embeddings)
    _, state_keep_indices = wfp.keep_closest_latent_samples(embeddings, states)

    print("[setup] loading diagnostics...")
    mass_flux = np.load(wfp.PATHS["data_raw"])[:-1]
    z_array = wfp.load_z_array()
    s_array = np.linspace(320, 350, mass_flux.shape[-1])
    pw = np.load(wfp.PATHS["pw"])[: len(states)]
    prec = np.load(wfp.PATHS["prec"])[: len(states)]

    n = min(len(mass_flux), len(states))
    mass_flux = mass_flux[:n]
    state_keep_indices_mf = {
        s: idxs[idxs < n]
        for s, idxs in state_keep_indices.items()
        if (idxs < n).any()
    }

    # Normalisation for the diagnostic colorings (mirrors K=7 notebook).
    pw_norm = Normalize(vmin=1, vmax=17)
    prec_norm = Normalize(vmin=0, vmax=6)

    # Transition matrix + stationary distribution.
    Tmat = wfp.compute_transition_matrix(states, NUM_STATES, future=5)
    pi = wfp.stationary_distribution(Tmat)

    print(f"[setup] N={len(states)}  mass_flux shape={mass_flux.shape}")
    print(f"[setup] cluster counts: "
          f"{dict(zip(*np.unique(states, return_counts=True)))}")
    print(f"[setup] pi = {[f'{p:.3f}' for p in pi]}")

    _cache.update(dict(
        loader=loader, input_dim=input_dim, model=model,
        embeddings=embeddings, states=states,
        emb_pca=emb_pca, pca=pca,
        state_keep_indices=state_keep_indices,
        state_keep_indices_mf=state_keep_indices_mf,
        mass_flux=mass_flux, z_array=z_array, s_array=s_array,
        pw=pw, prec=prec,
        pw_norm=pw_norm, prec_norm=prec_norm,
        Tmat=Tmat, pi=pi,
    ))
    return _cache


# SI-specific cache extension (loss history, 3D PCA, log-scaled data).
import pickle
from matplotlib.ticker import LogLocator
def _setup_si():
    """Augment the main cache with SI-specific data (loss history, 3D PCA,
    physical grids, log-scaled data). Cached on first call."""
    d = setup()
    if "loss_train" in d:
        return d

    print("[si setup] loading K=5 training history + physical grids...")
    exp_dir_k5 = os.path.join(ROOT, "data", "runs", "exp_20260223_1140_5c129e")
    with open(os.path.join(exp_dir_k5, "loss_history.pkl"), "rb") as f:
        loss_dict = pickle.load(f)

    def _parse_logs(logs):
        keys = ["total", "recon", "kl_scaled", "hmm_scaled",
                "entropy_unscaled", "trans_acc", "dice_loss_scaled"]
        return {k: np.array([row[k] for row in logs]) for k in keys}

    epochs = np.arange(len(loss_dict["train"]))
    loss_train = _parse_logs(loss_dict["train"])
    loss_val = _parse_logs(loss_dict["val"])

    # Physical grid + log-scaled data (used for reconstruction / canonical
    # representative panels).
    S, Z = wfp.load_physical_grids()
    data_log = np.load(wfp.PATHS["data_log"])
    if data_log.ndim > 2:
        data_log = data_log.reshape(len(data_log), -1)

    # 3-component PCA for pca_3d.
    from sklearn.decomposition import PCA as _PCA
    pca3 = _PCA(n_components=3)
    emb_pca_3d = pca3.fit_transform(d["embeddings"])

    d.update(dict(
        loss_train=loss_train,
        loss_val=loss_val,
        epochs=epochs,
        S=S, Z=Z,
        data_log=data_log,
        emb_pca_3d=emb_pca_3d,
        exp_dir_k5=exp_dir_k5,
    ))
    return d

# Run it now.
d = setup()

## Helpers

Loss-curve plotter (shared by training/validation_loss cells) and per-state autocorrelation cache (shared by fig9bis).

In [ ]:
def _plot_loss_curve(loss_series, epochs, out_path, title, ls):
    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    ax.plot(epochs, loss_series["total"], label="Total",
            lw=3.0, ls=ls, color="r", zorder=5)
    ax.plot(epochs, loss_series["recon"], label="Reconstruction",
            lw=2.2, ls=ls, color="#4C72B0")
    ax.plot(epochs, loss_series["kl_scaled"], label=r"KL ($\beta\,\mathrm{KL}$)",
            lw=2.2, ls=ls, color="#8FB1E3")
    ax.plot(epochs, loss_series["dice_loss_scaled"],
            label=r"Dice ($\lambda_{\mathrm{Dice}}$)",
            lw=2.2, ls=ls, color="#E1B12C")
    ax.plot(epochs, loss_series["hmm_scaled"],
            label=r"State transition ($\gamma_{\mathrm{MC}}$)",
            lw=2.2, ls=ls, color="#DD8452")
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=(0.2, 0.4, 0.6, 0.8)))
    ax.grid(which="major", alpha=0.35)
    ax.grid(which="minor", alpha=0.15)
    ax.legend(frameon=False, fontsize=10, loc="upper right", handlelength=2.8)
    ax.set_title(title, style="italic")
    plt.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)

TARGET_STATE_IDX_K5 = 0  # display State 1 = data index 0
SUBSAMPLE = 5
DT_LAG_MIN = 10
MAX_LAG_STEPS = 50

def _ensure_autocorrelation_cache(d):
    """Compute C_obs, C_markov_per_state, s_sub, cp_sub once and cache."""
    if "C_obs" in d:
        return
    states = d["states"]
    Tmat = d["Tmat"]
    pi = d["pi"]

    print("[fig9bis] computing per-state autocorrelations...")
    s_sub = states[::SUBSAMPLE]
    cp = np.load(wfp.PATHS["cpi"])[: len(states)]
    cp_sub = cp[::SUBSAMPLE]

    C_obs = wfp.indicator_autocorrelation_normalized(s_sub, max_lag=MAX_LAG_STEPS)
    C_markov_per_state = np.array([
        wfp.markov_autocorrelation_indicator(Tmat, pi, state_idx=s, max_lag=MAX_LAG_STEPS)
        for s in range(NUM_STATES)
    ])

    d["C_obs"] = C_obs
    d["C_markov_per_state"] = C_markov_per_state
    d["s_sub"] = s_sub
    d["cp_sub"] = cp_sub
    d["cp"] = cp

## Figure 3 — clusters in latent space

Function: `generate_fig3_clusters` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_fig3_clusters
generate_fig3_clusters()

## Figure 4 — latent axes vs physical diagnostics

Function: `generate_fig4_latent_axis` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_fig4_latent_axis
generate_fig4_latent_axis()

## Figure 5 — vertical mass-flux profiles

Function: `generate_fig5_vertical_profiles` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_fig5_vertical_profiles
generate_fig5_vertical_profiles()

## Figure 6 — mass-flux profiles along the MSE axis

Function: `generate_fig6_mse_profiles` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_fig6_mse_profiles
generate_fig6_mse_profiles()

## Figure 7 — Markov transition matrix + chain overlay

Function: `generate_fig7_markov_matrix` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_fig7_markov_matrix
generate_fig7_markov_matrix()

## Figure 9bis — Markovianity + cold-pool conditioning

Function: `generate_fig9bis_markovianity_coldpools` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_fig9bis_markovianity_coldpools
generate_fig9bis_markovianity_coldpools()

## Figure 8 (SI) — example trajectories

Function: `generate_fig8_trajectories` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_fig8_trajectories
generate_fig8_trajectories()

## SI — training loss

Function: `generate_training_loss` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_training_loss
generate_training_loss()

## SI — validation loss

Function: `generate_validation_loss` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_validation_loss
generate_validation_loss()

## SI — reconstruction examples

Function: `generate_reconstruction` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_reconstruction
generate_reconstruction()

## SI — canonical representative samples

Function: `generate_canonical_representative` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_canonical_representative
generate_canonical_representative()

## SI — 3D PCA projection

Function: `generate_pca_3d` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_pca_3d
generate_pca_3d()

## SI — cluster-count sensitivity (K=3, 7, 10)

Function: `generate_cluster_sensitivity` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_cluster_sensitivity
generate_cluster_sensitivity()

## SI — per-cluster mean 2D fields (LaCie)

Function: `generate_fields_clusters` in [figures_paper_5c.py](figures_paper_5c.py).

In [ ]:
from figures_paper_5c import generate_fields_clusters
generate_fields_clusters()